# 🗺️ 自分たちの街で量子配送 — Real City Edition

Tutorial で触った **短さの好み / 混ぜる強さ / 考え直す回数** を、**本物の地図** で動かす Colab です。

## こんなことができます
- 自分の街の 6 つの場所を入れる (例: 京都駅・清水寺・金閣寺…)
- 量子コンピュータが「6 箇所を最短で巡るルート」を波の干渉で見つける
- 結果が **本物の地図** 上にルートとして描かれる → ストーリーカードに貼れる

## 進め方
1. **③ チーム情報セル** の場所を自分の街に書き換える (or デフォルトの京都 6 選で進める)
2. メニュー: **ランタイム → すべて実行**
3. 3〜6 分待つ (Nominatim 検索 + Qiskit 計算)
4. ⑫ で出る地図をスクリーンショットしてストーリーカードに貼る

**所要時間**: 約 5 分
**前提**: 不要 (実行するだけで OK)

---

## ① 必要なライブラリをインストール
1〜2 分かかります。

In [ ]:
!pip install -q qiskit==1.2.4 qiskit-aer==0.15.1 folium geopy numpy pandas
print('✅ install 完了')

## ② チーム情報
右側のフォームで自由に書き換えできます。

In [ ]:
TEAM_NAME = "\u00d7\u00d7\u9ad8\u6821\u30c1\u30fc\u30e0" #@param {type:"string"}
CITY_NAME = "\u4eac\u90fd" #@param {type:"string"}
print(f'Team: {TEAM_NAME}')
print(f'City: {CITY_NAME}')

## ③ 6 つの配送先を入れる

- 例: 「京都駅, 京都市」「清水寺, 京都市」「東京駅, 東京都」など
- 「市」「区」など**場所を絞るキーワード**を入れると見つかりやすいです
- デフォルトは **京都 6 選** が入っています — そのまま実行可能

In [ ]:
PLACE_1 = "\u4eac\u90fd\u99c5, \u4eac\u90fd\u5e02" #@param {type:"string"}
PLACE_2 = "\u6e05\u6c34\u5bfa, \u4eac\u90fd\u5e02" #@param {type:"string"}
PLACE_3 = "\u91d1\u95a3\u5bfa, \u4eac\u90fd\u5e02" #@param {type:"string"}
PLACE_4 = "\u9280\u95a3\u5bfa, \u4eac\u90fd\u5e02" #@param {type:"string"}
PLACE_5 = "\u5d50\u5c71, \u4eac\u90fd\u5e02" #@param {type:"string"}
PLACE_6 = "\u4e8c\u6761\u57ce, \u4eac\u90fd\u5e02" #@param {type:"string"}

places = [PLACE_1, PLACE_2, PLACE_3, PLACE_4, PLACE_5, PLACE_6]
for i, p in enumerate(places, 1):
    print(f'{i}. {p}')

## ④ 地理座標を取得 (geopy / Nominatim)

場所名を緯度経度に変換します。Nominatim は無料の OpenStreetMap 検索 API。最大 6 件で 6〜10 秒。

In [ ]:
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent='quantum-hackathon-2026', timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

coords = []
for i, name in enumerate(places, 1):
    loc = geocode(name)
    if loc is None:
        raise RuntimeError(f'❌ \u898b\u3064\u304b\u308a\u307e\u305b\u3093: {name}\u3000\u300c\u5e02\u300d\u300c\u533a\u300d\u3092\u5165\u308c\u3066\u3082\u3046\u4e00\u5ea6\u8a66\u3057\u3066\u4e0b\u3055\u3044')
    coords.append({'name': name, 'lat': loc.latitude, 'lon': loc.longitude})
    print(f'{i}. {name}: ({loc.latitude:.4f}, {loc.longitude:.4f})')

# \u5009\u5eab = 6 \u70b9\u306e\u91cd\u5fc3
import numpy as np
depot_lat = float(np.mean([c['lat'] for c in coords]))
depot_lon = float(np.mean([c['lon'] for c in coords]))
print(f'\n\ud83c\udfe0 \u5009\u5eab (6 \u70b9\u306e\u91cd\u5fc3): ({depot_lat:.4f}, {depot_lon:.4f})')

## ⑤ 地図プレビュー

緑のピンが配送先、赤いピンが倉庫。

In [ ]:
import folium

preview_map = folium.Map(location=[depot_lat, depot_lon], zoom_start=12, tiles='OpenStreetMap')
folium.Marker(
    [depot_lat, depot_lon],
    popup=f'\ud83c\udfe0 \u5009\u5eab ({CITY_NAME})',
    icon=folium.Icon(color='red', icon='home', prefix='fa')
).add_to(preview_map)
for i, c in enumerate(coords, 1):
    folium.Marker(
        [c['lat'], c['lon']],
        popup=f"#{i}: {c['name']}",
        icon=folium.Icon(color='green', icon='box', prefix='fa')
    ).add_to(preview_map)
preview_map

## ⑥ 距離マトリックス (Haversine = 地球上の直線距離)

緯度経度から「地球の丸みを考慮した直線距離」を 6×6 で計算します。
(本物の道路距離ではなく**鳥瞰した直線**)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# 0 = depot, 1..6 = deliveries
all_points = [{'name': '\u5009\u5eab', 'lat': depot_lat, 'lon': depot_lon}] + coords
n_points = len(all_points)
dist = np.zeros((n_points, n_points))
for i in range(n_points):
    for j in range(i + 1, n_points):
        d = haversine_km(all_points[i]['lat'], all_points[i]['lon'], all_points[j]['lat'], all_points[j]['lon'])
        dist[i][j] = d
        dist[j][i] = d

import pandas as pd
labels = [p['name'][:12] for p in all_points]
print('距離マトリックス (km):')
print(pd.DataFrame(dist, index=labels, columns=labels).round(2))

## ⑦ 720 通りの順序 + ブルートフォース最短

6 つの配送先の巡回順序は 6! = 720 通り。古典総当たりで最短も求めておきます (Qiskit の答え合わせ用)。

In [ ]:
from itertools import permutations

delivery_indices = list(range(1, 7))
all_perms = list(permutations(delivery_indices))

def tour_cost(perm):
    total = dist[0][perm[0]]
    for a, b in zip(perm[:-1], perm[1:]):
        total += dist[a][b]
    total += dist[perm[-1]][0]
    return total

costs = np.array([tour_cost(p) for p in all_perms])
print(f'\ud83d\udcca {len(all_perms)} \u901a\u308a\u3001\u30b3\u30b9\u30c8\u7bc4\u56f2 = [{costs.min():.2f}, {costs.max():.2f}] km')

best_brute = int(np.argmin(costs))
brute_route = [coords[p - 1]['name'].split(',')[0] for p in all_perms[best_brute]]
print(f'\ud83c\udfc6 \u53e4\u5178\u7dcf\u5f53\u305f\u308a\u306e\u6700\u77ed: {costs[best_brute]:.2f} km')
print(f'        \u9806\u5e8f: \u5009\u5eab \u2192 {" \u2192 ".join(brute_route)} \u2192 \u5009\u5eab')

## ⑧ 量子のつまみ
Tutorial / Web アプリで触った値を入れてください。

In [ ]:
GAMMA = 1.0 #@param {type:"slider", min:0, max:3.14, step:0.05}
BETA = 0.4 #@param {type:"slider", min:0, max:1.57, step:0.02}
REPS = 2 #@param {type:"slider", min:1, max:3, step:1}
print(f'\u77ed\u3055\u306e\u597d\u307f \u03b3 = {GAMMA}')
print(f'\u6df7\u305c\u308b\u5f37\u3055   \u03b2 = {BETA}')
print(f'\u8003\u3048\u76f4\u3057\u56de\u6570 = {REPS}')

## ⑨ QAOA 回路 (Tutorial / Web アプリと同じロジック)

1. **重ね合わせ**: Hadamard で 1024 通りの状態を均等に出現
2. **位相付け (γ)**: 短いルートに有利な位相を回す対角ユニタリ
3. **混ぜる (β)**: Rx 回転で隣の状態と混ぜる
4. **2-3 を reps 回繰り返す**
5. **測定** で 1 つのルートを取り出す

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

num_qubits = int(np.ceil(np.log2(len(all_perms))))  # 10
size = 1 << num_qubits  # 1024
INVALID_PENALTY = 1e9

cost_vec = np.full(size, INVALID_PENALTY)
cost_vec[:len(costs)] = costs
valid_min, valid_max = costs.min(), costs.max()
span = max(valid_max - valid_min, 1e-9)
phases = np.empty(size)
for i in range(size):
    if cost_vec[i] >= INVALID_PENALTY:
        phases[i] = GAMMA * 1.2
    else:
        norm = (cost_vec[i] - valid_min) / span
        phases[i] = GAMMA * (1 - norm)
diag = np.exp(-1j * phases)

qc = QuantumCircuit(num_qubits)
qc.h(range(num_qubits))
for layer in range(REPS):
    qc.diagonal(diag.tolist(), list(range(num_qubits)))
    for q in range(num_qubits):
        qc.rx(2 * BETA, q)
qc.measure_all()

print(f'\u2705 {num_qubits} qubits / {REPS} \u30ec\u30a4\u30e4\u30fc / \u72b6\u614b\u7a7a\u9593 {size}')

## ⑩ AerSimulator で実行 (4096 shots)

In [ ]:
simulator = AerSimulator()
tqc = transpile(qc, simulator)
result = simulator.run(tqc, shots=4096).result()
counts = result.get_counts()

freq = np.zeros(size)
for bs, c in counts.items():
    freq[int(bs.replace(' ', ''), 2)] = c
freq /= freq.sum()
valid_freq = freq[:len(all_perms)]
valid_freq = valid_freq / max(valid_freq.sum(), 1e-12)
rank_indices = np.argsort(-valid_freq)

print('🌊 Qiskit 結果 (上位 5)')
for rank, idx in enumerate(rank_indices[:5], 1):
    perm = all_perms[idx]
    names = [coords[p - 1]['name'].split(',')[0] for p in perm]
    print(f'  #{rank}: 確率 {valid_freq[idx]*100:5.2f}% / 距離 {costs[idx]:6.2f} km')
    print(f'        \u5009\u5eab \u2192 {" \u2192 ".join(names)} \u2192 \u5009\u5eab')

## ⑪ 答え合わせ (古典総当たりとの一致)

In [ ]:
qiskit_top = int(rank_indices[0])
if qiskit_top == best_brute:
    print(f'\u2705 Qiskit \u3068\u53e4\u5178\u7dcf\u5f53\u305f\u308a\u304c\u4e00\u81f4! ({costs[best_brute]:.2f} km)')
else:
    gap_pct = (costs[qiskit_top] - costs[best_brute]) / costs[best_brute] * 100
    print(f'\u26a0 Qiskit 1 \u4f4d = {costs[qiskit_top]:.2f} km')
    print(f'  \u53e4\u5178\u6700\u9069   = {costs[best_brute]:.2f} km')
    print(f'  \u30ae\u30e3\u30c3\u30d7: +{gap_pct:.1f}% \u2014 reps \u3092\u5897\u3084\u3059 / \u03b3 \u30fb \u03b2 \u3092\u8abf\u6574\u3059\u308b\u3068\u6539\u5584\u3059\u308b\u53ef\u80fd\u6027\u3042\u308a')

## ⑫ 結果を地図に描画 — 📸 スクリーンショットしてストーリーカードへ

Qiskit が選んだ 1 位ルートを **シアン色のライン** で描きます。各ピンに巡回順番号付き。

In [ ]:
best_perm = all_perms[rank_indices[0]]
route_coords = [(depot_lat, depot_lon)]
for p in best_perm:
    route_coords.append((coords[p - 1]['lat'], coords[p - 1]['lon']))
route_coords.append((depot_lat, depot_lon))

result_map = folium.Map(location=[depot_lat, depot_lon], zoom_start=12, tiles='OpenStreetMap')
folium.Marker(
    [depot_lat, depot_lon],
    popup=f'\ud83c\udfe0 \u5009\u5eab ({CITY_NAME}) — {TEAM_NAME}',
    icon=folium.Icon(color='red', icon='home', prefix='fa')
).add_to(result_map)
for visit_idx, p in enumerate(best_perm, 1):
    c = coords[p - 1]
    folium.Marker(
        [c['lat'], c['lon']],
        popup=f"#{visit_idx}: {c['name']}",
        tooltip=f"#{visit_idx}",
        icon=folium.Icon(color='green', icon='box', prefix='fa')
    ).add_to(result_map)
folium.PolyLine(
    route_coords,
    color='#3DD9C8',
    weight=5,
    opacity=0.85,
    popup=f'Qiskit \u6700\u9069\u30eb\u30fc\u30c8 ({costs[rank_indices[0]]:.2f} km) — \u03b3={GAMMA}, \u03b2={BETA}, reps={REPS}'
).add_to(result_map)
result_map

## 🎯 まとめ — チームの成果物にする

1. **このノートブックを共有可能にする**: メニュー > 共有 > 「リンクを取得」を「閲覧者」で
2. **⑫ の地図をスクショ**: ブラウザのフルスクショ or 切り取りツール
3. **ストーリーカード PDF に貼る** (Web アプリの /story から印刷)

### プレゼンで言いたいこと
- 私たちは **どんな街** で **どんな問題** を解いた? (ストーリーカード参照)
- つまみは γ=___, β=___, reps=___ で「ちょうど良い」を探した
- Qiskit と古典総当たりが **一致** した? ギャップは何 km / 何 %?
- もし配送先が 10 個になったら? (720 → 362 万通り = 量子の出番)

---

### 🔭 もっと先へ
- 配送先を 7-10 個に増やす (state space 2^N でメモリ注意)
- IBM Quantum 実機で動かす ([quantum.ibm.com](https://quantum.ibm.com/))
- 道路距離 (osmnx) を使って本物の最短経路にする